# Inspecting Retrocomposite reaction rules

Adapted from SynPlanner's `tutorials/12_Rule_Analysis.ipynb`. The original popularity,
composition, center, ring, duplicate and comparison analyses follow the inspection browser.

**Start here:** use the **Python (retrocomposite_env)** kernel and run the first cells in order.
The default is the frozen single-step dataset, **rule 4500 / policy row 31189 (n1 route 434)**.
Then use the browser to select another rule and its training examples. Every rule ID and
policy row index is **zero-based in the selected TSV**, with no renumbering after filtering.

The notebook reads local artifacts. It does not train, change datasets, install packages,
or download presets. All chemistry and drawings use **Chython**, with SmilesDrawer for 2D layout.
The saved notebook has no stale outputs; run it to generate views from your selected files.

On another machine, restore the dataset following [the mobility guide](../docs/MOBILITY.md).
The normalization audit is computed from the release's own files; the earlier local
investigation report and a separate SynPlanner/RouteInspector checkout are not needed.

In [1]:
from pathlib import Path
import csv
import json
import sys
from functools import lru_cache
from hashlib import sha256
from html import escape
from collections import Counter, defaultdict

import chython
from chython import smiles
from IPython.display import HTML, display
import pandas as pd
from synplan.chem.reaction.rules.analysis import RuleSet
from synplan.chem.reaction.rules.representation import (
    cgr_from_reaction_rule, reaction_query_to_reaction,
)
from synplan.chem.reaction.reactor import apply_reaction_rule
from synplan.chem.utils import unite_molecules, canonical_query_cgr_key
from synplan.utils.loading import load_reaction_rules, _parse_reaction_rule
from retrocomposite.identity import molecule_identity

# Keep the existing chemical state; depiction never calls canonicalize().
chython.clean2d_engine = "smilesdrawer"

PROJECT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                if (p / "retrocomposite").is_dir()), None)
if PROJECT is None:
    raise RuntimeError("Start Jupyter from the retrocomposite project or its notebooks directory.")

RELEASE = PROJECT / "data/releases/n1-n5-composites-inchi-v1"
DATA_ROOT = RELEASE if RELEASE.is_dir() else PROJECT / "data"
DATASET = "single_step"  # alternatives: "primary", "patent_disjoint"

ASSETS = DATA_ROOT / ("single_step" if DATASET == "single_step" else f"cohorts/{DATASET}")
STEM = "single_step" if DATASET == "single_step" else "composite"
RULES_PATH = ASSETS / f"{STEM}.tsv"
POLICY_PATH = ASSETS / f"{STEM}_policy_data.tsv"
for path in (RULES_PATH, POLICY_PATH):
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}. Restore the release or change DATA_ROOT / DATASET above.")
print("Python:", sys.executable)
print("Dataset:", DATASET)
print("Rules:", RULES_PATH)
print("Rules SHA-256:", sha256(RULES_PATH.read_bytes()).hexdigest())

Python: /Users/almazgil/miniforge3-arm/envs/retrocomposite_env/bin/python
Dataset: single_step
Rules: /Users/almazgil/Developer/projects/retrocomposite/data/releases/n1-n5-composites-inchi-v1/single_step/single_step.tsv
Rules SHA-256: 479d34a98d8e670f889f6ee29405a802d432a97bd1a4280d0b9bdd98642d3ac5


In [2]:
def read_tsv(path):
    with path.open(newline="") as stream:
        return list(csv.DictReader(stream, delimiter="\t"))


def read_jsonl(path):
    with path.open() as stream:
        for line in stream:
            yield json.loads(line)


def strict_ruleset(path):
    rows = read_tsv(path)
    parsed = []
    for rule_id, row in enumerate(rows):
        try:
            # The pinned SynPlanner loader handles component-local CXSMARTS here.
            parsed.append(_parse_reaction_rule(row["rule_smarts"]))
        except Exception as error:
            raise ValueError(f"Rule {rule_id} failed to parse in {path}; IDs were not renumbered") from error
    return RuleSet(parsed, [r["rule_smarts"] for r in rows],
                   [int(r["popularity"]) for r in rows],
                   [tuple(int(i) for i in r["reaction_indices"].split(",") if i) for r in rows])


ruleset = strict_ruleset(RULES_PATH)
policy = read_tsv(POLICY_PATH)
if not ruleset or not policy:
    raise ValueError("Empty vocabulary or policy data")
rows_by_rule = defaultdict(list)
for row_index, row in enumerate(policy):
    rule_id = int(row["rule_id"])
    if not 0 <= rule_id < len(ruleset):
        raise ValueError(f"Policy row {row_index} refers to missing rule {rule_id}")
    rows_by_rule[rule_id].append(row_index)

# Build the same reactors as planning, including symmetry handling.
reactors = load_reaction_rules(str(RULES_PATH))
assert len(reactors) == len(ruleset)
native_path = ASSETS / f"{STEM}_policy_data.native.tsv"
native_policy = read_tsv(native_path) if native_path.is_file() else None
ledger_path = ASSETS / "policy_product_normalization.jsonl"
ledger = list(read_jsonl(ledger_path)) if ledger_path.is_file() else []
if native_policy is not None:
    assert len(native_policy) == len(policy), "Native/exported row counts differ"
    actual_changes = []
    for i, (old, new) in enumerate(zip(native_policy, policy, strict=True)):
        assert old["rule_id"] == new["rule_id"], f"Rule label changed at row {i}"
        if old["product_smiles"] != new["product_smiles"]:
            actual_changes.append(dict(row_index=i, rule_id=int(new["rule_id"]),
                                       original=old["product_smiles"], canonical=new["product_smiles"]))
    assert actual_changes == ledger, "Normalization ledger does not match the selected policy files"

provenance_path = ASSETS / "reaction_provenance.jsonl"
provenance = {r["input_index"]: r for r in read_jsonl(provenance_path)} if provenance_path.is_file() else {}
composite_path = ASSETS / "composite_records.jsonl"
composite_records = defaultdict(list)
if composite_path.is_file():
    for record in read_jsonl(composite_path):
        composite_records[record["training_pair_index"]].append(record)
split_path = DATA_ROOT / "splits" / DATASET / "assignments.jsonl"
assignments = {}
if split_path.is_file():
    candidate = {r["row_index"]: r for r in read_jsonl(split_path)}
    if len(candidate) == len(policy) and all(
        candidate.get(i, {}).get("product_smiles") == row["product_smiles"]
        and candidate[i]["rule_id"] == int(row["rule_id"]) for i, row in enumerate(policy)
    ):
        assignments = candidate
    else:
        print("Split assignments do not match these policy rows; partition labels are omitted.")
print(f"Loaded {len(ruleset):,} rules, {len(policy):,} policy rows; {len(ledger)} normalization changes.")

Loaded 11,209 rules, 57,381 policy rows; 22 normalization changes.


## Normalization failures to inspect

The investigation found two keto–enol InChI changes and 16 lost rule applications among
22 normalized single-step rows in the original release. The table below **recomputes** these
observations for your selected files. Equal stereo-free InChIs do not guarantee equal
rule applicability. A successful replay also needs its precursors checked against provenance.

Only rows in the normalization ledger are audited here; this is not a whole-corpus replay test.
Counts are capped at 100 outcomes per input, and diagnostics are retained. No-outcome results
with a mapping-budget diagnostic are inconclusive. A corrected release can have different IDs
and counts; use the live table, rather than assuming rule 4500 always denotes this chemistry.

In [3]:
@lru_cache(maxsize=512)
def replay(rule_id, product_smiles):
    diagnostics = []
    outcomes = list(apply_reaction_rule(smiles(product_smiles), reactors[rule_id],
                                       top_reactions_num=100, diagnostics=diagnostics))
    return outcomes, diagnostics


audit_rows = []
for change in ledger:
    i, rule_id = change["row_index"], change["rule_id"]
    original, original_diag = replay(rule_id, change["original"])
    prepared, prepared_diag = replay(rule_id, change["canonical"])
    audit_rows.append({
        "policy_row": i, "rule_id": rule_id,
        "inchi_changed": molecule_identity(change["original"]) != molecule_identity(change["canonical"]),
        "original_outcomes": len(original), "exported_outcomes": len(prepared),
        "lost_applicability": bool(original) and not prepared and not prepared_diag,
        "partition": assignments.get(i, {}).get("partition", "unknown"),
        "diagnostics": json.dumps({"original": original_diag, "exported": prepared_diag}),
    })
audit = pd.DataFrame(audit_rows)
problem_rule_ids = sorted({r["rule_id"] for r in audit_rows if r["lost_applicability"]})
if audit_rows:
    display(audit)
    print("Changed InChIs:", sum(r["inchi_changed"] for r in audit_rows))
    print("Lost applicability:", sum(r["lost_applicability"] for r in audit_rows))
    print("Rules to inspect:", problem_rule_ids)
else:
    print("No normalization ledger rows for this dataset. Browse any rule below.")

,policy_row,rule_id,inchi_changed,original_outcomes,exported_outcomes,lost_applicability,partition,diagnostics
0,2571,3321,False,1,1,False,validation,"{""original"": [], ""exported"": []}"
1,6753,3564,False,1,3,False,train,"{""original"": [], ""exported"": []}"
2,7161,6070,False,1,0,True,train,"{""original"": [], ""exported"": []}"
3,13591,6686,False,1,0,True,train,"{""original"": [], ""exported"": []}"
4,14381,6757,False,1,0,True,train,"{""original"": [], ""exported"": []}"
5,18182,4075,False,1,0,True,train,"{""original"": [], ""exported"": []}"
6,21798,609,False,1,0,True,train,"{""original"": [], ""exported"": []}"
7,23314,7533,False,1,0,True,train,"{""original"": [], ""exported"": []}"
8,24711,253,False,1,0,True,train,"{""original"": [], ""exported"": []}"
9,31188,4500,True,1,0,True,train,"{""original"": [], ""exported"": []}"


Changed InChIs: 2
Lost applicability: 16
Rules to inspect: [253, 609, 4075, 4500, 4610, 6070, 6686, 6757, 7533, 10315]


## Visual rule and training-example browser

The rule arrow points from a target to retrosynthetic precursors. Source-reaction arrows
follow the recorded forward direction. Rule-template atom numbers and source-reaction atom
numbers belong to different mappings. Target panels hide atom numbers to avoid implying alignment.

The browser displays the exact stored native/exported structures; layout changes coordinates
only. It shows raw source reactions and replayed precursor sets. InChI-based precursor agreement
is a molecule-identity comparison, not an exact mapped-graph certificate.

In [4]:
def structure_svg(structure, *, mapped=False):
    item = structure.copy()
    item.clean2d()
    chython.depict_settings(aam=mapped)
    try:
        return item.depict()
    finally:
        chython.depict_settings(aam=True)


def panels(items, *, mapped=False):
    blocks = []
    for label, structure in items:
        try:
            drawing = structure_svg(structure, mapped=mapped)
        except Exception as error:
            drawing = f"<pre>Drawing failed: {escape(str(error))}</pre>"
        blocks.append(f"<div style='flex:1;min-width:300px'><b>{escape(label)}</b>{drawing}</div>")
    display(HTML("<style>.rc-structures svg {max-width:100%;height:auto;background:white}</style>"
                 "<div class='rc-structures' style='display:flex;flex-wrap:wrap;gap:20px'>"
                 + "".join(blocks) + "</div>"))


def details(label, text):
    display(HTML(f"<details><summary>{escape(label)}</summary>"
                 f"<pre style='white-space:pre-wrap'>{escape(text)}</pre></details>"))


def show_rules(indices):
    for rule_id in indices:
        print(f"Rule {rule_id} | popularity {ruleset.popularity[rule_id]} | "
              f"{len(rows_by_rule[rule_id])} policy rows")
        panels([(f"Rule {rule_id} — retro", ruleset[rule_id])], mapped=True)
        details("Exact rule SMARTS", ruleset.smarts_strings[rule_id])


@lru_cache(maxsize=256)
def sources_for_row(row_index):
    if not provenance:
        return []
    row = policy[row_index]
    rule_id = int(row["rule_id"])
    original = (native_policy or policy)[row_index]["product_smiles"]
    identity = molecule_identity(original)
    matches = []
    # Match molecule identity AND rule support; policy-row order is not a provenance join.
    for index in ruleset.reaction_indices[rule_id]:
        record = provenance[index]
        reaction = smiles(record["mapped"])
        if molecule_identity(unite_molecules(reaction.products)) == identity:
            matches.append(record)
    return matches


def precursor_ids(molecules):
    return sorted(molecule_identity(m) for m in molecules)


def show_example(row_index):
    row = policy[row_index]
    rule_id = int(row["rule_id"])
    print(f"Policy row {row_index} | rule {rule_id} | "
          f"partition: {assignments.get(row_index, {}).get('partition', 'unknown')}")
    exported = row["product_smiles"]
    native = native_policy[row_index]["product_smiles"] if native_policy else None
    inputs = ([] if native is None else [("Original extraction target", native)])
    inputs.append(("Exported model input", exported))
    panels([(label, smiles(text)) for label, text in inputs])
    identifiers = [{"state": label, "smiles": text, "stereo_free_inchi": molecule_identity(text)}
                   for label, text in inputs]
    display(pd.DataFrame(identifiers))
    details("Full input SMILES and stereo-free InChIs", json.dumps(identifiers, indent=2))
    if native is not None:
        print("Same stereo-free InChI:", molecule_identity(native) == molecule_identity(exported))

    source_records = sources_for_row(row_index)
    expected = [precursor_ids(smiles(r["mapped"]).reactants) for r in source_records]
    if source_records:
        display(pd.DataFrame([{"input_index": r["input_index"], **s}
                              for r in source_records for s in r["sources"]]))
        print(f"{len(source_records)} supporting reaction(s); showing up to 5.")
        for r in source_records[:5]:
            panels([(f"Recorded source reaction {r['input_index']} — forward", smiles(r["mapped"]))], mapped=True)
    elif provenance:
        print("No source join found: investigate provenance before interpreting the label.")
    for r in composite_records.get(row_index, []):
        details("Composite occurrence and decomposition", json.dumps(r, indent=2))

    for label, text in inputs:
        outcomes, diagnostics = replay(rule_id, text)
        matches = any(precursor_ids(result) in expected for result in outcomes) if expected else None
        print(f"{label}: {len(outcomes)} outcome(s), cap 100; "
              f"matches recorded precursor InChIs: {matches if matches is not None else 'not assessed'}")
        if diagnostics:
            display(pd.DataFrame(diagnostics))
        for j, result in enumerate(outcomes[:3]):
            panels([(f"{label} / outcome {j + 1} / precursor {k + 1}", m)
                    for k, m in enumerate(result)])
        if len(outcomes) > 3:
            print("Only the first 3 outcomes are drawn; all retained outcomes were compared.")

In [5]:
# Change these and rerun this cell for a simple, non-widget inspection.
RULE_ID = 4500 if DATASET == "single_step" and len(ruleset) > 4500 else 0
example_rows = rows_by_rule[RULE_ID]
POLICY_ROW = 31189 if DATASET == "single_step" and 31189 in example_rows else (example_rows[0] if example_rows else None)
show_rules([RULE_ID])
if POLICY_ROW is not None:
    show_example(POLICY_ROW)

Rule 4500 | popularity 2 | 2 policy rows


Policy row 31189 | rule 4500 | partition: train


,state,smiles,stereo_free_inchi
0,Original extraction target,c1ccccc1C23c4nc(-c5ccccc5OC)nc(c4CCC2C(C)C(C(C...,InChI=1S/C28H28N2O4/c1-17-22-14-13-21-25(28(22...
1,Exported model input,c1cc(ccc1)C23C(CCc4c2nc(nc4OC)-c5ccccc5OC)C(C)...,InChI=1S/C28H28N2O4/c1-17-22-14-13-21-25(28(22...


Same stereo-free InChI: False


,input_index,dataset,node_path,reaction_id,route_id
0,31189,n1,"[0, 0, 0]",US20150225397A1;0956;1736983,434
1,31189,n5,"[0, 0, 0]",US20150225397A1;0956;1736983,5079


1 supporting reaction(s); showing up to 5.


Original extraction target: 1 outcome(s), cap 100; matches recorded precursor InChIs: True


Exported model input: 0 outcome(s), cap 100; matches recorded precursor InChIs: False


In [6]:
import ipywidgets as widgets

rule_picker = widgets.BoundedIntText(value=RULE_ID, min=0, max=len(ruleset)-1, description="Rule ID:")
problem_picker = widgets.Dropdown(options=[("Choose a flagged rule", None)] + [(str(i), i) for i in problem_rule_ids],
                                  description="Flagged:")
row_picker = widgets.Dropdown(options=rows_by_rule[RULE_ID], value=POLICY_ROW, description="Policy row:")
inspect_button = widgets.Button(description="Inspect selection", button_style="primary")
inspection_output = widgets.Output()


def refresh_rows(change):
    rows = rows_by_rule[change["new"]]
    row_picker.options = rows
    row_picker.value = rows[0] if rows else None


def choose_problem(change):
    if change["new"] is not None:
        rule_picker.value = change["new"]


def inspect_selection(_):
    with inspection_output:
        inspection_output.clear_output(wait=True)
        show_rules([rule_picker.value])
        if row_picker.value is not None:
            show_example(row_picker.value)


rule_picker.observe(refresh_rows, names="value")
problem_picker.observe(choose_problem, names="value")
inspect_button.on_click(inspect_selection)
display(widgets.VBox([widgets.HBox([problem_picker, rule_picker]),
                      widgets.HBox([row_picker, inspect_button]), inspection_output]))

## Vocabulary overview

The following sections retain the SynPlanner tutorial analyses for the selected vocabulary.
These can take longer than inspecting one rule. Drawings keep original vocabulary IDs.
Ring/center statistics use concrete projections of query rules and are descriptive diagnostics;
conversion failures are reported and excluded from the relevant counts.

In [ ]:
show_rules(range(min(5, len(ruleset))))

## 1. Popularity Overview

Reaction rules follow a long-tail distribution: a small fraction of rules covers most reactions. Understanding this distribution helps decide the `min_popularity` threshold for extraction and the number of rules needed for planning.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

pop = np.array(ruleset.popularity)

print(f"Total rules: {len(pop)}")
print(f"Median popularity: {np.median(pop):.0f}")
print(f"90th percentile: {np.percentile(pop, 90):.0f}")
print(f"99th percentile: {np.percentile(pop, 99):.0f}")
print(f"Max popularity: {pop.max()}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Histogram (log scale)
axes[0].hist(pop, bins=50, edgecolor="black", log=True)
axes[0].set_xlabel("Popularity")
axes[0].set_ylabel("Number of rules (log)")
axes[0].set_title("Popularity distribution")

# Rank-frequency (log-log)
sorted_pop = np.sort(pop)[::-1]
axes[1].plot(range(1, len(sorted_pop) + 1), sorted_pop)
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_xlabel("Rule rank")
axes[1].set_ylabel("Popularity")
axes[1].set_title("Rank-frequency (Zipf)")

# Cumulative coverage
cumulative = np.cumsum(sorted_pop) / sorted_pop.sum() * 100
axes[2].plot(range(1, len(cumulative) + 1), cumulative)
axes[2].axhline(80, color="red", linestyle="--", alpha=0.5, label="80% coverage")
n_80 = np.searchsorted(cumulative, 80) + 1
axes[2].axvline(n_80, color="red", linestyle="--", alpha=0.5)
axes[2].set_xlabel("Number of rules")
axes[2].set_ylabel("Cumulative coverage (%)")
axes[2].set_title(f"Pareto: {n_80} rules cover 80%")
axes[2].legend()

plt.tight_layout()
plt.show()

## 2. Product Count Distribution

Each rule has a fixed number of products (retrosynthetic precursors). Most rules produce 1-2 products. Rules with many products are worth inspecting — they may represent complex multi-component reactions.

In [ ]:
from collections import Counter

n_products = [len(rule.products) for rule in ruleset]
dist = Counter(n_products)

plt.figure(figsize=(7, 4))
plt.bar(dist.keys(), dist.values(), color="#2a9d8f", edgecolor="black")
plt.xlabel("Number of products")
plt.ylabel("Number of rules")
plt.title("Product count distribution")
plt.xticks(sorted(dist.keys()))
plt.show()

print(f"Distribution: {dict(sorted(dist.items()))}")

In [ ]:
# Inspect rules with the most products
k = max(n_products)
indices_k = [i for i, rule in enumerate(ruleset.rules) if len(rule.products) == k]
print(f"{len(indices_k)} rules with {k} products:")
show_rules(indices_k[:5])

## 3. Element Composition

Rules may contain atoms beyond the common organic set (C, N, O, H). Identifying rules with exotic elements (metals, metalloids) helps understand rule coverage and potential issues with building block availability.

In [ ]:
from collections import Counter

from synplan.chem.reaction.rules.representation import query_to_mol

BASE_ATOMS = {"C", "N", "O", "H", "S", "Br", "Cl", "B", "I", "F", "P", "Si"}

element_counts = Counter()
rules_with_special = []

for i, rule in enumerate(ruleset):
    elements = set()
    for mol in (*rule.reactants, *rule.products):
        m = query_to_mol(mol)
        for _, atom in m.atoms():
            elements.add(atom.atomic_symbol)
    special = elements - BASE_ATOMS
    if special:
        rules_with_special.append((i, rule, special))
        element_counts.update(special)

print(
    f"{len(rules_with_special)} / {len(ruleset)} rules contain special atoms "
    f"({len(rules_with_special) * 100 / len(ruleset):.1f}%)"
)

if element_counts:
    elems, counts = zip(*element_counts.most_common())
    plt.figure(figsize=(10, 4))
    plt.bar(elems, counts, color="#e76f51", edgecolor="black")
    plt.xlabel("Element")
    plt.ylabel("Number of rules")
    plt.title("Rules containing special (non-base) atoms")
    plt.show()

In [ ]:
# Inspect first 10 rules with special atoms
special_indices = [idx for idx, _, _ in rules_with_special[:10]]
print(f"Showing {len(special_indices)} rules with special atoms:")
show_rules(special_indices)

## 4. Reaction Center Size

The reaction center consists of atoms and bonds that actually change during the reaction. Larger centers mean more specific rules. We extract this information from the Condensed Graph of Reaction (CGR).

In [ ]:
from tqdm.auto import tqdm

center_atoms_list = []
center_bonds_list = []
failed = []

for i, rule in enumerate(tqdm(ruleset, desc="Analyzing centers")):
    try:
        cgr = cgr_from_reaction_rule(rule)
        center_atoms_list.append(len(cgr.center_atoms))
        center_bonds_list.append(len(cgr.center_bonds))
    except Exception:
        failed.append(i)
        center_atoms_list.append(None)
        center_bonds_list.append(None)

if failed:
    print(f"{len(failed)} CGR failures excluded; rule IDs: {failed}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

atoms_dist = Counter(x for x in center_atoms_list if x is not None)
axes[0].bar(atoms_dist.keys(), atoms_dist.values(), edgecolor="black")
axes[0].set_xlabel("Center atoms")
axes[0].set_ylabel("Number of rules")
axes[0].set_title("Reaction center atom count")

bonds_dist = Counter(x for x in center_bonds_list if x is not None)
axes[1].bar(bonds_dist.keys(), bonds_dist.values(), edgecolor="black")
axes[1].set_xlabel("Center bonds")
axes[1].set_ylabel("Number of rules")
axes[1].set_title("Reaction center bond count")

plt.tight_layout()
plt.show()

## 5. Ring & Aromatic Transformations

Rules can create or open rings, change ring sizes, and alter aromaticity. This section analyzes how often these transformations occur across the rule set.

In [ ]:
from tqdm.auto import tqdm


def ring_bond_set(molecules):
    """Return set of (min, max) atom-pair tuples for all SSSR ring bonds."""
    bonds = set()
    for mol in molecules:
        for ring in mol.sssr:
            ring = list(ring)
            for a, b in zip(ring, [*ring[1:], ring[0]]):
                bonds.add((min(a, b), max(a, b)))
    return bonds


def aromatic_bond_set(molecules):
    """Return set of (min, max) atom-pair tuples for aromatic bonds."""
    bonds = set()
    for mol in molecules:
        for i, j, bond in mol.bonds():
            if int(bond) == 4:
                bonds.add((min(i, j), max(i, j)))
    return bonds


ring_failures = []
ring_created_list = []
ring_opened_list = []
arom_changed_list = []

for _i, rule in enumerate(tqdm(ruleset, desc="Analyzing rings")):
    try:
        rxn = reaction_query_to_reaction(rule)
        r_rings = ring_bond_set(rxn.reactants)
        p_rings = ring_bond_set(rxn.products)
        ring_created_list.append(bool(p_rings - r_rings))
        ring_opened_list.append(bool(r_rings - p_rings))

        r_arom = aromatic_bond_set(rxn.reactants)
        p_arom = aromatic_bond_set(rxn.products)
        arom_changed_list.append(r_arom != p_arom)
    except Exception as error:
        ring_failures.append((_i, str(error)))
        ring_created_list.append(None)
        ring_opened_list.append(None)
        arom_changed_list.append(None)

events = {
    "Ring created": sum(x for x in ring_created_list if x is not None),
    "Ring opened": sum(x for x in ring_opened_list if x is not None),
    "Aromaticity changed": sum(x for x in arom_changed_list if x is not None),
}

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(events.keys(), events.values(), edgecolor="black")
ax.set_ylabel("Number of rules")
ax.set_title("Ring and aromaticity events")
plt.show()

print("Excluded ring-analysis failures:", ring_failures)
valid_ring_count = len(ruleset) - len(ring_failures)
for name, count in events.items():
    print(f"{name}: {count} ({count * 100 / max(valid_ring_count, 1):.1f}%)")

In [ ]:
creator_indices = [i for i, created in enumerate(ring_created_list) if created]
print(f"{len(creator_indices)} ring-creating rules. First 5:")
show_rules(creator_indices[:5])

## 6. Multi-Center Rules

Some rules have disconnected reaction centers — multiple independent transformations happening in the same rule. These are often multi-component reactions (e.g., double acylation).

In [ ]:
from tqdm.auto import tqdm

multi_center_failures = []
multi_center_data = []

for _i, rule in enumerate(tqdm(ruleset, desc="Detecting multi-centers")):
    try:
        cgr = cgr_from_reaction_rule(rule)
        centers = cgr.centers_list
        n_centers = len(centers)
    except Exception as error:
        multi_center_failures.append((_i, str(error)))
        n_centers = None
    multi_center_data.append(n_centers)

center_dist = Counter(x for x in multi_center_data if x is not None)
n_multi = sum(1 for nc in multi_center_data if nc is not None and nc > 1)

plt.figure(figsize=(7, 4))
plt.bar(center_dist.keys(), center_dist.values(), edgecolor="black")
plt.xlabel("Number of disconnected centers")
plt.ylabel("Number of rules")
plt.title("Multi-center rule distribution")
plt.xticks(sorted(center_dist.keys()))
plt.show()

print(f"{n_multi} multi-center rules ({n_multi * 100 / max(len(ruleset) - len(multi_center_failures), 1):.1f}%)")
print("Excluded multi-center failures:", multi_center_failures)

In [ ]:
multi_indices = [i for i, nc in enumerate(multi_center_data) if nc is not None and nc > 1]
print("First 5 multi-center rules:")
show_rules(multi_indices[:5])

## 7. Duplicate Detection

Compare exact stored SMARTS strings and SynPlanner's canonical **QueryCGR keys**, which retain
query constraints. Failed conversions are listed separately. These diagnostics concern rule
representation, not molecule InChI identity or E3 occurrence exclusion.

In [ ]:
smarts_groups = defaultdict(list)
for i, text in enumerate(ruleset.smarts_strings):
    smarts_groups[text].append(i)
smarts_dupes = {key: ids for key, ids in smarts_groups.items() if len(ids) > 1}

cgr_groups = defaultdict(list)
duplicate_failures = []
for i, rule in enumerate(tqdm(ruleset, desc="Canonical QueryCGR keys")):
    try:
        cgr_groups[canonical_query_cgr_key(~rule)].append(i)
    except Exception as error:
        duplicate_failures.append((i, str(error)))
cgr_dupes = {key: ids for key, ids in cgr_groups.items() if len(ids) > 1}
print(f"Stored SMARTS: {len(smarts_groups)} unique; {len(smarts_dupes)} duplicate groups")
print(f"QueryCGR: {len(cgr_groups)} unique among {len(ruleset)-len(duplicate_failures)} converted rules; "
      f"{len(cgr_dupes)} duplicate groups")
print("Excluded QueryCGR failures:", duplicate_failures)

In [ ]:
if cgr_dupes:
    top_dupes = sorted(cgr_dupes.values(), key=len, reverse=True)[:5]
    for group in top_dupes:
        print(f"Duplicate group ({len(group)} rules): indices {group}")
    # Show one representative rule from each duplicate group
    rep_indices = [
        sorted(cgr_dupes.values(), key=len, reverse=True)[i][0]
        for i in range(min(5, len(cgr_dupes)))
    ]
    show_rules(rep_indices)

## 8. Comparing Two Rule Sets

When tuning extraction parameters, it is useful to compare rule sets extracted under different configurations. This section finds rules unique to each set.

In [ ]:
# Optional: set a project-relative path to another vocabulary, then rerun.
OTHER_RULES = None  # e.g. DATA_ROOT / "cohorts/primary/composite.tsv"
if OTHER_RULES is not None:
    path_b = Path(OTHER_RULES)
    if not path_b.is_absolute():
        path_b = PROJECT / path_b
    ruleset_b = strict_ruleset(path_b)
    def keyed_rules(collection):
        keys, errors = defaultdict(list), []
        for i, rule in enumerate(tqdm(collection, desc="Comparing QueryCGR keys")):
            try:
                keys[canonical_query_cgr_key(~rule)].append(i)
            except Exception as error:
                errors.append((i, str(error)))
        return keys, errors
    keys_a, errors_a = keyed_rules(ruleset)
    keys_b, errors_b = keyed_rules(ruleset_b)
    print("Shared keys:", len(keys_a.keys() & keys_b.keys()))
    print("Only in A:", len(keys_a.keys() - keys_b.keys()))
    print("Only in B:", len(keys_b.keys() - keys_a.keys()))
    print("Excluded failures A / B:", errors_a, errors_b)
    only_a_ids = [ids[0] for key, ids in keys_a.items() if key not in keys_b]
    show_rules(only_a_ids[:5])
else:
    print("Set OTHER_RULES above to compare another vocabulary.")

## Source and license

Adapted from `SynPlanner/tutorials/12_Rule_Analysis.ipynb` (local source snapshot recorded in
this notebook's metadata). Copyright © 2024 Laboratoire-de-Chemoinformatique; MIT license.
See [the retained license](LICENSE.SynPlanner). The original notebook is preserved in SynPlanner.

Retrocomposite additions: portable dataset paths, exact vocabulary IDs, normalization-ledger
checks, source provenance, stereo-free InChIs, live replay, Chython-only drawings, and a rule browser.

In [2]:
from chython import smiles, depict_settings

depict_settings(aam=False)

smiles('COc1ccc2nc(-c3ccc(OC)c(F)c3)nc(C(=O)NC(CO)Cc3c[nH]c4cccnc34)c2c1')

In [3]:
smiles('CC(C)(C)OC(=O)NC1(c2nc(O)c3c(Cl)ccn3n2)CC1')